In [ ]:
pip install pandas

In [ ]:
import pandas as pd

In [ ]:
pip install requests

In [ ]:
import requests
import json
import os
from datetime import datetime


In [ ]:
OUTPUT_DIR="intel_corpus"

In [ ]:
EPSS_THRESHOLD = 0.05
MITRE_STIX_URL = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json"
CISA_KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
EPSS_API_URL = "https://api.first.org/data/v1/epss"
NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
NVD_API_KEY = None  # Optional: Set your NVD API key here if you have one

In [ ]:
try:
    response = requests.get(CISA_KEV_URL, timeout=15)
    response.raise_for_status()
    data = response.json()
    # Extract CVE IDs into a set for fast lookup
    kev_cves = {vuln['cveID'] for vuln in data.get('vulnerabilities', [])}
    print(f"[+] Successfully loaded {len(kev_cves)} CVEs from CISA KEV.")
except Exception as e:
    print(f"[-] Error fetching CISA KEV: {e}")
    kev_cves = None

In [ ]:
kev_cves

Fetches EPSS scores from FIRST API for a specific list of CVEs. Batches requests to handle API limits smoothly.

In [ ]:

print("[*] Fetching EPSS Scores...")
epss_mapping = {}
    
# Convert set/list to list and chunk into groups of 100 to avoid long URLs
cve_list = list(kev_cves)
chunk_size = 100

for i in range(0, len(cve_list), chunk_size):
    chunk = cve_list[i:i + chunk_size]
    cves_param = ",".join(chunk)
    
    try:
        response = requests.get(EPSS_API_URL, params={"cve": cves_param}, timeout=15)
        response.raise_for_status()
        results = response.json().get('data', [])
        
        for item in results:
            epss_mapping[item['cve']] = {
                "epss": float(item['epss']),
                "percentile": float(item['percentile'])
            }
    except Exception as e:
        print(f"[-] Error fetching EPSS chunk starting at index {i}: {e}")
        continue
        
print(f"[+] Successfully fetched EPSS data for {len(epss_mapping)} CVEs.")

Fetches the MITRE ATT&CK Enterprise Matrix in STIX 2.1 format.
Extracts unstructured behavioral text profiles for Threat Groups (Intrusion Sets).

In [ ]:

print("[*] Fetching MITRE ATT&CK STIX Data...")
attack_profiles = []
try:
    response = requests.get(MITRE_STIX_URL, timeout=30)
    response.raise_for_status()
    stix_data = response.json()
    
    # Parse STIX objects to extract Intrusion Sets (APT Groups)
    for obj in stix_data.get('objects', []):
        if obj.get('type') == 'intrusion-set':
            profile = {
                "group_id": obj.get('id'),
                "group_name": obj.get('name'),
                "description": obj.get('description', ''),
                "aliases": obj.get('aliases', []),
                "last_updated": obj.get('modified')
            }
            attack_profiles.append(profile)
            
    print(f"[+] Successfully extracted {len(attack_profiles)} MITRE Threat Group narratives.")
    # return attack_profiles
except Exception as e:
    print(f"[-] Error fetching MITRE ATT&CK: {e}")

Queries the official NIST NVD API v2.0 to extract the raw unstructured textual description of a specific CVE.


In [ ]:
def fetch_nvd_cve_description(cve_id):
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY
        
    params = {"cveId": cve_id}
    
    try:
        response = requests.get(NVD_API_URL, headers=headers, params=params, timeout=15)
        if response.status_code == 403:
            print(f"[!] Rate limit hit on NVD API for {cve_id}. Sleeping and retrying...")
            time.sleep(6)  # NVD requires strict windows without an API key
            response = requests.get(NVD_API_URL, headers=headers, params=params, timeout=15)
            
        response.raise_for_status()
        data = response.json()
        
        # Dig into the NVD JSON response structure to pull the English text
        vulnerabilities = data.get("vulnerabilities", [])
        if vulnerabilities:
            cve_item = vulnerabilities[0].get("cve", {})
            descriptions = cve_item.get("descriptions", [])
            for desc in descriptions:
                if desc.get("lang") == "en":
                    return desc.get("value", "")
                    
        return "No English description found in NVD records."
    except Exception as e:
        print(f"[-] Failed to fetch NVD description for {cve_id}: {e}")
        return "Failed to fetch description from source."

Extracts raw threat narratives for Threat Groups from MITRE ATT&CK STIX.

In [12]:
print("[*] Fetching MITRE ATT&CK STIX Data...")
try:
    response = requests.get(MITRE_STIX_URL, timeout=30)
    response.raise_for_status()
    stix_data = response.json()
    
    attack_profiles = []
    for obj in stix_data.get('objects', []):
        if obj.get('type') == 'intrusion-set':
            profile = {
                "group_id": obj.get('id'),
                "group_name": obj.get('name'),
                "description": obj.get('description', ''),
                "aliases": obj.get('aliases', []),
                "last_updated": obj.get('modified')
            }
            attack_profiles.append(profile)
    # return attack_profiles
except Exception as e:
    print(f"[-] Error fetching MITRE ATT&CK: {e}")
    # return []

KeyboardInterrupt: 

Executes the pipeline, maps elements together, and creates the validation metrics.

In [ ]:
print("=== Starting Ingestion Pipeline ===")
import time
if not kev_cves:
    # Emergency baseline sample for testing if CISA endpoint drops out
    kev_cves = {"CVE-2021-44228", "CVE-2019-11510", "CVE-2023-38831"}

epss_data =  epss_mapping
mitre_groups = attack_profiles

# Process and build prioritized Vulnerability Corpus
vulnerability_corpus = {}


# target_cves = list(kev_cves)[0:10]

target_cves = list(kev_cves) [0:100] 
print(f"[*] Querying NVD descriptions for a sample batch of {len(target_cves)} CVEs...")

for idx, cve in enumerate(target_cves):
    print(f"    [{idx+1}/{len(target_cves)}] Querying NVD text for: {cve}")
    
    # 1. Fetch text from NVD
    description_text = fetch_nvd_cve_description(cve)
    
    score_info = epss_data.get(cve, {"epss": 0.0, "percentile": 0.0})
    
    # 2. Append directly to structured entry
    vulnerability_corpus[cve] = {
        "cve_id": cve,
        "description": description_text,  # 🌟 Here is your NLP training description
        "is_known_exploited": True,
        "epss_score": score_info['epss'],
        "epss_percentile": score_info['percentile'],
        "ingested_at": datetime.utcnow().isoformat()
    }
    
    # Mandatory polite throttling delay to prevent NVD API blacklisting
    time.sleep(3 if NVD_API_KEY else 6)
        
# Export Data
vuln_output_path = os.path.join(OUTPUT_DIR, "validated_vulnerabilities.json")
with open(vuln_output_path, "w") as f:
    json.dump(vulnerability_corpus, f, indent=4)
    
mitre_output_path = os.path.join(OUTPUT_DIR, "mitre_group_narratives.json")
with open(mitre_output_path, "w") as f:
    json.dump(mitre_groups, f, indent=4)
    
print("\n       Ingestion Pipeline Complete     ")
print(f"Saved updated CVE corpus with NVD descriptions to: {vuln_output_path}")
print(f"Saved MITRE threat group narratives to: {mitre_output_path}")

=== Starting Ingestion Pipeline ===
[*] Querying NVD descriptions for a sample batch of 100 CVEs...
    [1/100] Querying NVD text for: CVE-2024-55591
    [2/100] Querying NVD text for: CVE-2015-4902
    [3/100] Querying NVD text for: CVE-2020-3452
    [4/100] Querying NVD text for: CVE-2024-12987
    [5/100] Querying NVD text for: CVE-2021-34484
    [6/100] Querying NVD text for: CVE-2005-2773
    [7/100] Querying NVD text for: CVE-2013-3896
    [8/100] Querying NVD text for: CVE-2026-20133
    [9/100] Querying NVD text for: CVE-2016-3351
    [10/100] Querying NVD text for: CVE-2021-22941
    [11/100] Querying NVD text for: CVE-2021-34523
    [12/100] Querying NVD text for: CVE-2021-26855
    [13/100] Querying NVD text for: CVE-2023-20963
    [14/100] Querying NVD text for: CVE-2017-6316
    [15/100] Querying NVD text for: CVE-2023-21492
    [16/100] Querying NVD text for: CVE-2020-0638
    [17/100] Querying NVD text for: CVE-2016-20017
    [18/100] Querying NVD text for: CVE-2023-1671